In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Limpieza y Deduplicación -> Capa Silver
Aplicación de reglas de limpieza y guardado en formato Delta.

In [0]:
df_user = spark.table('castor.bronze.brz_fact_users')
df_trans = spark.table('castor.bronze.brz_fact_transactions')

In [0]:
# Limpieza usuarios
# Deduplicar por id_usuario tomando el más reciente (si hubiera un timestamp, aquí se usa row_number())
df_user_clean = df_user.dropDuplicates(['id_usuario'])

# Imputación de nulos según README
df_user_clean = df_user_clean.fillna({'pais': 'Desconocido'})
mediana_edad = df_user_clean.approxQuantile('edad', [0.5], 0.01)[0]
df_user_clean = df_user_clean.fillna({'edad': mediana_edad})

display(df_user_clean.limit(5))

In [0]:
# Limpieza transacciones
df_trans_clean = df_trans.withColumn('monto_clean', F.regexp_replace(F.col('monto'), r'[\$,\s]', '').cast('double')).withColumn('fecha_transaccion', F.to_date(F.from_unixtime('timestamp_unix')))

df_trans_clean = df_trans_clean.fillna({'categoria': 'Sin categoría'})
display(df_trans_clean.limit(5))

In [0]:
df_user_clean.write.mode('overwrite').saveAsTable('castor.silver.slv_fact_users')
display(df_user_clean)

In [0]:
df_trans_clean.write.mode('overwrite').saveAsTable('castor.silver.slv_fact_transactions')
display(df_trans_clean)